In [5]:
import os
import pandas as pd
from sqlalchemy import create_engine
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

# 1. Cargamos las variables del archivo .env
load_dotenv(override=True)

pd.options.display.float_format = '{:.0f}'.format

# 2. Configuramos el cliente de IA (Groq) de forma global
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("LLM_API_KEY")
)

# 3. Configuramos el motor de MySQL de forma global
engine = create_engine(
    f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

In [6]:
def consultar_biblioia(pregunta_usuario):
    prompt_sistema = """
    Sos un asistente experto en bases de datos MySQL para una biblioteca. 
    Tu objetivo es traducir preguntas en lenguaje natural a consultas SQL precisas.

    ESQUEMA DE LA BASE DE DATOS:
    - Libro (Isbn, Titulo, Año, StockTotal, StockDisponible)
    - Socio (Id, Dni, Nombre, Apellido, Mail, FechaAlta, IdSexo, IdEstadoSocio)
    - Ejemplar (Numero,FechaAlta, IsbnLibro, IdEstadoEjemplar)
    - Prestamo (Id,FechaPrestamo, FechaVencimiento, FechaDevolucion, IdSocio, IdEjemplar, IdEstadoPrestamo)
    - Sancion (Id, FechaInicio, FechaFin, Motivo, IdTipoSancion, IdPrestamo, IdSocio)
    - GeneroLibro(Id, Nombre, Descripcion)
    - Autor(Id, Nombre, Apellido,IdSexo, IdNacionalidad)
    - Nacionalidad(Id, Pais)
    - Autor_Libro(IdAutor, IsbnLibro)
    - GeneroLibro_Libro(IsbnLibro, IdGeneroLibro)
    - TipoSancion(Id, Tipo)
    - Sexo(Id, Tipo)
    - EstadoSocio(Id, Tipo)
    - EstadoEjemplar(Id, Tipo)
    - EstadoPrestamo(Id, Tipo)
    
    CONSTRAINTS Y ESTADOS:
    - EstadoSocio: 1='Activo', 2='Suspendido', 3='Baja'
    - EstadoEjemplar: 1='Disponible', 2='Prestado', 3='Baja'
    - EstadoPrestamo: 1='Activo', 2='Devuelto', 3='Vencido'
    - TipoSancion: 1='Leve', 2='Medio', 3='Grave'
    - Sexo:  1='Femenino',2='Masculino', 3='Otro'

    INSTRUCCIÓN CRÍTICA:
    Respondé ÚNICA Y EXCLUSIVAMENTE con una consulta SQL válida para MySQL.
    Devuelve solo la cadena de texto de la consulta, sin bloques de código Markdown ni explicaciones.
    """
    
    try:
        chat_completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": pregunta_usuario}
            ],
            temperature=0.1
        )
        sql_generado = chat_completion.choices[0].message.content.strip()
        return sql_generado.replace("```sql", "").replace("```", "").strip()
    except Exception as e:
        return f"Error al generar SQL: {e}"

def ejecutar_consulta(sql):
    try:
        with engine.connect() as conexion:
            df = pd.read_sql(sql, conexion)
        return df
    except Exception as e:
        return f"Error de MySQL: {e}"

def agente_responder(pregunta):
    print("🤖 Pensando y generando SQL...")
    sql_generado = consultar_biblioia(pregunta)
    
    if sql_generado.startswith("Error"):
        print(f"❌ Hubo un problema al generar el código SQL: {sql_generado}")
        return
        
    print("📊 Buscando datos en la base de datos...\n")
    resultado = ejecutar_consulta(sql_generado)
    
    if isinstance(resultado, pd.DataFrame):
        if resultado.empty:
            print("⚠️ La consulta fue exitosa, pero no encontró datos.")
        else:
            resultado.index = resultado.index + 1
            display(resultado)
    else:
        print(f"❌ {resultado}")

In [7]:
def obtener_datos_para_recomendacion(id_o_dni_socio):
    query_socio = "SELECT Id, Nombre, Apellido FROM Socio WHERE Id = %s OR Dni = %s LIMIT 1;"
    
    with engine.connect() as conexion:
        socio_df = pd.read_sql(query_socio, conexion, params=(id_o_dni_socio, id_o_dni_socio))
        if socio_df.empty:
            return None, None
        
        socio_id = int(socio_df.iloc[0]['Id'])
        socio_nombre = f"{socio_df.iloc[0]['Nombre']} {socio_df.iloc[0]['Apellido']}"
        
        query_libros = f"""
        SELECT DISTINCT l.Isbn, l.Titulo, l.Año,
               (SELECT GROUP_CONCAT(DISTINCT CONCAT(a.Nombre, ' ', a.Apellido) SEPARATOR ', ')
                FROM Autor_Libro al JOIN Autor a ON al.IdAutor = a.Id WHERE al.IsbnLibro = l.Isbn) AS Autores,
               (SELECT GROUP_CONCAT(DISTINCT g.Nombre SEPARATOR ', ')
                FROM GeneroLibro_Libro gll JOIN GeneroLibro g ON gll.IdGeneroLibro = g.Id WHERE gll.IsbnLibro = l.Isbn) AS Generos
        FROM Libro l
        LEFT JOIN Autor_Libro al_filtro ON l.Isbn = al_filtro.IsbnLibro
        LEFT JOIN GeneroLibro_Libro gll_filtro ON l.Isbn = gll_filtro.IsbnLibro
        WHERE l.StockDisponible > 0 
          AND (
               al_filtro.IdAutor IN (
                   SELECT DISTINCT al2.IdAutor FROM Prestamo p
                   JOIN Ejemplar e ON p.IdEjemplar = e.Numero
                   JOIN Autor_Libro al2 ON e.IsbnLibro = al2.IsbnLibro
                   WHERE p.IdSocio = {socio_id}
               )
               OR 
               gll_filtro.IdGeneroLibro IN (
                   SELECT DISTINCT gll2.IdGeneroLibro FROM Prestamo p
                   JOIN Ejemplar e ON p.IdEjemplar = e.Numero
                   JOIN GeneroLibro_Libro gll2 ON e.IsbnLibro = gll2.IsbnLibro
                   WHERE p.IdSocio = {socio_id}
               )
          )
          AND l.Isbn NOT IN (
              SELECT DISTINCT e3.IsbnLibro FROM Prestamo p3
              JOIN Ejemplar e3 ON p3.IdEjemplar = e3.Numero
              WHERE p3.IdSocio = {socio_id}
          )
        LIMIT 4;
        """
        libros_df = pd.read_sql(query_libros, conexion)
        
    return socio_nombre, libros_df

def generar_recomendaciones_ia(socio_nombre, df_libros):
    if df_libros.empty:
        return f"### ¡Hola {socio_nombre}! 📚\nActualmente no encontramos libros disponibles que coincidan con tu historial. ¡Te invitamos a explorar nuevas categorías!"
    
    lista_libros = "\n".join([f"- {row['Titulo']} | Autores: {row['Autores']} | Géneros: {row['Generos']}" for _, row in df_libros.iterrows()])
    
    prompt_sistema = """
    Sos el bibliotecario virtual de BiblioIA. Elegí hasta 3 libros de la lista provista y redactá una recomendación en Markdown:
    
    #### 📕 [TÍTULO DEL LIBRO]
    * **Portada Descriptiva:** [Describí conceptual y visualmente una portada moderna para este libro].
    * **Por qué te va a encantar:** [Justificá la elección conectándola con los gustos del socio].
    """
    
    try:
        chat_completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": f"Socio: {socio_nombre}\n\nLibros candidatos:\n{lista_libros}"}
            ],
            temperature=0.7
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"❌ Error en el LLM: {e}"

def modulo_recomendaciones(id_o_dni):
    nombre, df = obtener_datos_para_recomendacion(id_o_dni)
    
    if not nombre:
        print("❌ Error: No se encontró un socio con ese ID o DNI.")
        return
        
    print(f"✨ Procesando gustos de: {nombre}...")
    resultado = generar_recomendaciones_ia(nombre, df)
    display(Markdown(f"## 📚 SUGERENCIAS PARA: {nombre.upper()}\n---\n" + resultado))

In [ ]:
def iniciar_demo():
    print("="*60)
    print("📚🤖 INTERFAZ DE PRUEBA - BIBLIO-IA 🤖📚")
    print(" • Consultas generales: Escribí normal (ej: '¿Cuántos socios hay?')")
    print(" • Recomendaciones: Escribí exactamente -> recomendar [DNI o ID]")
    print(" • Salir: Escribí 'salir'")
    print("="*60)
    
    while True:
        entrada = input("\n👤 Tu consulta: ").strip()
        
        if not entrada:
            continue
            
        if entrada.lower() in ['salir', 'exit', 'quit']:
            print("🤖 ¡Nos vemos! Finalizando sesión.")
            break
            
        elif entrada.lower().startswith("recomendar"):
            partes = entrada.split()
            if len(partes) > 1:
                try:
                    id_o_dni_ingresado = int(partes[1])
                    modulo_recomendaciones(id_o_dni_ingresado)
                except ValueError:
                    print("❌ Formato inválido. Ejemplo: recomendar 12345678")
            else:
                print("❌ Especificá el número. Ejemplo: recomendar 1")
                
        else:
            agente_responder(entrada)

# Lanzar la demo
iniciar_demo()